# Customer Discovery & Stakeholder Communication — Reference Patterns — Hands-On

**FDE Delivery · Week 23a**

Offline notebook: engagement lifecycle, discovery question bank, validated discovery brief, confidence-weighted MVP scope evaluator, three-option executive tradeoff table, and a mini executive summary.

## 0. FDE engagement lifecycle

```mermaid
flowchart LR
  D[Discovery] --> R[Requirements]
  R --> S[MVP Scope]
  S --> A[Architecture]
  A --> B[Build]
  B --> DEP[Deploy]
  DEP --> O[Operate]
  O --> H[Handoff]
  O -. eval regression .-> R
  O -. stakeholder sync .-> D
```

The loop is deliberate: eval regressions, stakeholder feedback, and risk changes flow back into requirements and scope instead of surprising the customer later.

In [ ]:
from enum import Enum
from typing import Literal
import re
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

print('Imports ready: pydantic models will make FDE templates executable.')

## 1. The eight discovery questions

In [ ]:
questions = [
    ('Business problem','What metric or decision must improve?','Prevents solving a novelty problem.'),
    ('User','Who does the work and who adopts the tool?','Defines workflow, UX, and training.'),
    ('Current workflow','How does work happen now, including exceptions?','Finds integration and trust requirements.'),
    ('Data','Where is the evidence and who owns access?','Data readiness gates the MVP.'),
    ('Success','What target, metric, and timeframe define value?','Turns enthusiasm into acceptance.'),
    ('Constraints','What technical, regulatory, budget, or timeline limits apply?','Avoids impossible scope.'),
    ('Risks','What could cause failure or harm?','Creates mitigation owners early.'),
    ('Must not break','Which workflows, controls, or relationships are non-negotiable?','Protects production trust.'),
]
for q, ask, why in questions:
    print(f'{q:18s} | {ask:58s} | WHY: {why}')

## 2. DiscoveryBrief generator for the medical-coding scenario

In [ ]:
class Influence(str, Enum): low='low'; high='high'
class Interest(str, Enum): low='low'; high='high'
class Stakeholder(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str; role: str; interest: Interest; influence: Influence; concern: str
    @property
    def quadrant(self):
        if self.interest == Interest.high and self.influence == Influence.high: return 'Manage Closely'
        if self.interest == Interest.low and self.influence == Influence.high: return 'Keep Satisfied'
        if self.interest == Interest.high and self.influence == Influence.low: return 'Keep Informed'
        return 'Monitor'
class SuccessCriterion(BaseModel):
    model_config = ConfigDict(extra='forbid')
    statement: str; metric: str; target: str; timeframe: str
    @model_validator(mode='after')
    def smart_enough(self):
        if not (self.metric and re.search(r'\d', self.target) and re.search(r'week|month|quarter|pilot|by|year', self.timeframe.lower())):
            raise ValueError('SMART criteria need metric, numeric target, and timeframe')
        return self
class Constraints(BaseModel):
    model_config = ConfigDict(extra='forbid')
    technical: list[str]; regulatory: list[str]; budget: list[str]; timeline: list[str]
class Risk(BaseModel):
    model_config = ConfigDict(extra='forbid')
    risk: str; likelihood: Literal['low','medium','high']; impact: Literal['low','medium','high']; mitigation: str; owner: str
class DiscoveryBrief(BaseModel):
    model_config = ConfigDict(extra='forbid')
    customer: str; vague_ask: str; business_problem: str; user: str; current_workflow: str
    data_locations: list[str]; success_criteria: list[SuccessCriterion]; constraints: Constraints; risks: list[Risk]
    must_not_break: list[str]; jtbd: str; stakeholders: list[Stakeholder]; five_whys: list[str]
    @field_validator('jtbd')
    @classmethod
    def jtbd_shape(cls, v):
        if not re.match(r'^When .+, I want to .+, so I can .+\.?$', v.strip()): raise ValueError('bad JTBD shape')
        return v
    def render_brief_markdown(self):
        lines = [f'# Discovery Brief: {self.customer}', f'Vague ask: {self.vague_ask}', '', '## Answers']
        lines += [f'- Business problem: {self.business_problem}', f'- User: {self.user}', f'- Workflow: {self.current_workflow}', f'- Data: {"; ".join(self.data_locations)}']
        lines += ['- Success: ' + '; '.join(c.statement for c in self.success_criteria), '- Must not break: ' + '; '.join(self.must_not_break), '', '## JTBD', self.jtbd, '', '## Stakeholders']
        lines += [f'- {s.name} ({s.role}) => {s.quadrant}: {s.concern}' for s in self.stakeholders]
        lines += ['', '## Risks'] + [f'- {r.risk} [{r.likelihood}/{r.impact}] owner={r.owner}; mitigation={r.mitigation}' for r in self.risks]
        return '\n'.join(lines)

In [ ]:
brief = DiscoveryBrief(
 customer='Northstar Health Provider Network', vague_ask='We want an AI assistant for medical coding.',
 business_problem='Complex ambulatory encounters take too long to code and coding-related denials are increasing.',
 user='Certified medical coders working specialty clinic encounters.',
 current_workflow='Coders read encounter notes, search payer guidance, choose ICD-10 and CPT codes, add modifiers, and route ambiguous cases to a lead coder.',
 data_locations=['Epic export tables','payer policy PDFs','historical coded claims','denial reason reports'],
 success_criteria=[SuccessCriterion(statement='Reduce average coding research time.', metric='minutes per complex encounter', target='18 to 11 minutes', timeframe='within 10-week pilot'), SuccessCriterion(statement='Maintain grounded recommendations.', metric='golden-set groundedness', target='>= 92%', timeframe='before pilot go-live')],
 constraints=Constraints(technical=['nightly batch only','no EHR writeback v1'], regulatory=['HIPAA PHI boundary','audit trail'], budget=['55 engineering days','<$0.08 per call'], timeline=['scope in 2 weeks','demo in 8 weeks']),
 risks=[Risk(risk='Rare specialty groundedness regression', likelihood='medium', impact='high', mitigation='SME golden set and low-confidence review', owner='coding SME'), Risk(risk='PHI in traces', likelihood='low', impact='high', mitigation='redacted tracing', owner='security')],
 must_not_break=['coder sign-off workflow','PHI access controls','audit of final codes'],
 jtbd='When I am coding a complex specialty encounter, I want to see grounded code candidates with evidence, so I can finish accurately with less research and fewer denials.',
 stakeholders=[Stakeholder(name='VP Revenue Cycle', role='sponsor', interest='high', influence='high', concern='denial rate'), Stakeholder(name='Senior Medical Coder', role='end user', interest='high', influence='low', concern='trust and speed'), Stakeholder(name='Security Architect', role='security', interest='low', influence='high', concern='HIPAA boundary')],
 five_whys=['AI assistant because research is slow','Research is slow because evidence is scattered','Scattered evidence causes inconsistent coding','Inconsistency increases denials','Pilot targets high-denial specialties'])
print(brief.render_brief_markdown())

## 3. MVP scope and tradeoff evaluator

In [ ]:
REACH = {'low':1,'medium':2,'high':3,'enterprise':5}
class FeatureCandidate(BaseModel):
    model_config = ConfigDict(extra='forbid')
    name: str; description: str; business_value_score: int = Field(ge=1, le=10); ai_confidence: float = Field(ge=0, le=1); effort_days: int = Field(gt=0); reach: Literal['low','medium','high','enterprise']; must_have_reason_or_null: str | None = None; dependencies: list[str] = Field(default_factory=list)
    @property
    def score(self): return (self.business_value_score * self.ai_confidence * REACH[self.reach]) / self.effort_days

def score_and_partition(candidates, timebox_days):
    ordered = sorted(candidates, key=lambda c: (c.must_have_reason_or_null is not None, c.score), reverse=True)
    must = [c for c in ordered if c.must_have_reason_or_null]; remaining = [c for c in ordered if not c.must_have_reason_or_null]
    used = sum(c.effort_days for c in must); should=[]; could=[]; wont=[]
    for c in remaining:
        if used + c.effort_days <= timebox_days and c.ai_confidence >= .65: should.append(c); used += c.effort_days
        elif c.ai_confidence >= .55 and c.effort_days <= 8: could.append(c)
        else: wont.append(c)
    def pack(label, features, rationale):
        days=sum(f.effort_days for f in features); risky=[f.name for f in features if f.ai_confidence < .70]
        return (label, [f.name for f in features], days, days-timebox_days, risky, rationale)
    return {'MoSCoW': {'Must':must,'Should':should,'Could':could,"Won't":wont}, 'options': [pack('A aggressive', must+should+could[:3], 'Max capability, higher delivery risk'), pack('B recommended', must+should, 'Best risk-adjusted MVP'), pack('C safe', must+[c for c in should if c.ai_confidence>=.80 and not c.dependencies], 'Narrower and more certain')]}

In [ ]:
candidates = [
 FeatureCandidate(name='Encounter note summarization',description='Summarize diagnoses and ambiguity.',business_value_score=8,ai_confidence=.82,effort_days=7,reach='high',must_have_reason_or_null='core'),
 FeatureCandidate(name='ICD-10 and CPT candidate suggestions',description='Return codes with explanations.',business_value_score=10,ai_confidence=.74,effort_days=12,reach='high',must_have_reason_or_null='core'),
 FeatureCandidate(name='Guideline citation retrieval',description='Cite payer policy.',business_value_score=9,ai_confidence=.86,effort_days=9,reach='high',must_have_reason_or_null='trust'),
 FeatureCandidate(name='Golden-set eval harness',description='Score groundedness.',business_value_score=9,ai_confidence=.88,effort_days=6,reach='medium',must_have_reason_or_null='eval gate'),
 FeatureCandidate(name='Coder feedback capture',description='Capture accepted and rejected suggestions.',business_value_score=7,ai_confidence=.90,effort_days=5,reach='high'),
 FeatureCandidate(name='Audit log and trace export',description='Record prompt, sources, user decision.',business_value_score=8,ai_confidence=.84,effort_days=6,reach='enterprise'),
 FeatureCandidate(name='Nightly batch claims ingestion',description='Load historical claims and denials.',business_value_score=7,ai_confidence=.78,effort_days=8,reach='medium'),
 FeatureCandidate(name='Real-time EHR writeback',description='Write final codes to EHR.',business_value_score=8,ai_confidence=.40,effort_days=18,reach='high',dependencies=['vendor approval']),
 FeatureCandidate(name='Denial risk prediction',description='Predict denial probability.',business_value_score=7,ai_confidence=.52,effort_days=10,reach='medium',dependencies=['labels']),
 FeatureCandidate(name='Voice dictation workflow',description='Dictate corrections.',business_value_score=4,ai_confidence=.70,effort_days=7,reach='low')]
result = score_and_partition(candidates, 55)
for bucket, feats in result['MoSCoW'].items(): print(bucket, [f.name for f in feats])
for label, features, days, delta, risky, rationale in result['options']:
    print(f'\n{label}: {days}d ({delta:+d}) | {rationale}\n features={features}\n risky={risky or "none"}')

## 4. Worked three-options tradeoff table and mini executive summary

In [ ]:
tradeoffs = [
    ('Option A aggressive','62 days','Includes denial prediction and voice workflow','Higher AI uncertainty and over timebox'),
    ('Option B recommended','53 days','Core suggestions, citations, eval, audit, feedback','Defers writeback and denial prediction'),
    ('Option C safe','40 days','Core trusted assistant only','Less visible automation, fastest trust path'),
]
for row in tradeoffs:
    print(f'{row[0]:22s} | effort={row[1]:7s} | upside={row[2]:55s} | tradeoff={row[3]}')
print('\nBLUF: Recommend Option B for the 8-week pilot. It targets a reduction from 18 to 11 minutes per complex encounter while requiring groundedness >= 92%, audit logging, and <$0.08 average cost per call.')
print('Situation: coding time and denial rework are rising in specialty clinics.')
print('Complication: evidence is scattered across EHR notes, payer PDFs, and historical claims; real-time writeback is not available.')
print('Question: can we reduce research time without increasing compliance risk?')
print('Answer: yes, approve the balanced MVP scope, unblock nightly data access, and assign coding SME review for the golden set.')

## Exercises
1. Rewrite the JTBD for a different persona, such as a coding operations manager.
2. Add a RACI table for prompt changes, eval regressions, and incident response.
3. Add a new feature candidate and explain why it lands in Should, Could, or Won't.
4. Convert the mini executive summary into a 90-second one-pager.

## Links
- Literature note: `02 Literature Notes/FDE Delivery/Customer Discovery & Stakeholder Communication — Reference Patterns`
- Snippets: `04 Code Snippets/FDE Delivery/FDE Week 23a Discovery Brief Generator`, `.../FDE Week 23a MVP Scope and Tradeoff Evaluator`
- MOC: `06 Maps of Content/FDE Delivery Concepts`